<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/19-statistics.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 19 — The Statistics You Actually Need

Companion to [the chapter](https://www.ai.biz/books/python-primer/statistics/).

Everything here is demonstrated by simulation. Seeing it happen a thousand times
is more convincing than a formula.


In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(0)


## 1. Sampling variation

A world where the true mean is exactly 50.


In [ ]:
truth = 50
means = [rng.normal(truth, 15, 30).mean() for _ in range(1000)]
print(f'true mean          : {truth}')
print(f'average of samples : {np.mean(means):.2f}')
print(f'spread of samples  : {np.std(means):.2f}')
print(f'range seen         : {min(means):.1f} to {max(means):.1f}')
print()
print('Any one of those samples, alone, would have looked like a solid finding.')


## 2. The spread falls with the SQUARE ROOT of n


In [ ]:
for n in [10, 30, 100, 400, 1600]:
    sd = np.std([rng.normal(truth, 15, n).mean() for _ in range(1000)])
    print(f'n={n:>5}: spread = {sd:.3f}')
print()
print('Four times the data halves the wobble. It does not quarter it.')


## 3. The bootstrap

Resample your own data to simulate the variation you would have seen.


In [ ]:
data = rng.normal(50, 15, 200)
boot = [rng.choice(data, len(data), replace=True).mean() for _ in range(10_000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f'estimate     : {data.mean():.2f}')
print(f'95% interval : {lo:.2f} to {hi:.2f}')


It works for almost any statistic. One line changes.


In [ ]:
for name, fn in [('median', np.median),
                 ('90th pct', lambda x: np.percentile(x, 90)),
                 ('std dev', np.std)]:
    b = [fn(rng.choice(data, len(data), replace=True)) for _ in range(2000)]
    lo, hi = np.percentile(b, [2.5, 97.5])
    print(f'{name:<10}: {fn(data):6.2f}  [{lo:.2f}, {hi:.2f}]')
print()
print('Try finding the standard formula for a 90th percentile interval.')


## 4. What a confidence interval actually means

Not '95% chance the truth is in here'. It is a property of the *method*.


In [ ]:
contained = 0
for _ in range(500):
    sample = rng.normal(truth, 15, 30)
    b = [rng.choice(sample, 30, replace=True).mean() for _ in range(300)]
    lo, hi = np.percentile(b, [2.5, 97.5])
    contained += (lo <= truth <= hi)
print(f'{contained/5:.1f}% of intervals contained the truth')
print('That is what 95% confidence means.')


## 5. A p-value, by permutation


In [ ]:
a = rng.normal(50, 15, 100)
b = rng.normal(52, 15, 100)
observed = b.mean() - a.mean()

pooled = np.concatenate([a, b])
null = []
for _ in range(10_000):
    s = rng.permutation(pooled)
    null.append(s[100:].mean() - s[:100].mean())

p = np.mean(np.abs(null) >= abs(observed))
print(f'observed difference : {observed:.2f}')
print(f'p-value             : {p:.4f}')
print()
print('Shuffling destroys the group structure. This measures what chance produces.')


## 6. Effect size matters more than significance


In [ ]:
for n in [50, 500, 5000, 50_000]:
    a = rng.normal(50, 15, n); b = rng.normal(50.3, 15, n)   # tiny real difference
    pooled = np.concatenate([a, b])
    null = [np.abs(rng.permutation(pooled)[n:].mean() - rng.permutation(pooled)[:n].mean())
            for _ in range(500)]
    p = np.mean(null >= abs(b.mean() - a.mean()))
    d = (b.mean() - a.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1))/2)
    print(f'n={n:>6}: p={p:.3f}  effect size={d:.3f}')
print()
print('The effect never changes. Only the p-value does.')


## 7. Multiple comparisons

Test twenty things at 5% and expect one false positive.


In [ ]:
runs_with_false_positive = 0
for _ in range(300):
    groups = [rng.normal(50, 15, 30) for _ in range(20)]   # ALL from the same world
    base = groups[0]
    for g in groups[1:]:
        pooled = np.concatenate([base, g])
        null = [abs(rng.permutation(pooled)[30:].mean() - rng.permutation(pooled)[:30].mean())
                for _ in range(100)]
        if np.mean(np.array(null) >= abs(g.mean() - base.mean())) < 0.05:
            runs_with_false_positive += 1
            break
print(f'{runs_with_false_positive/3:.0f}% of runs found a "significant" result')
print('There were no real effects at all.')


## 8. Simpson's paradox


In [ ]:
dept = pd.DataFrame({
    'dept':   ['A','A','B','B'],
    'sex':    ['men','women','men','women'],
    'applied':[400, 100, 100, 400],
    'rate':   [0.60, 0.65, 0.20, 0.25],
})
dept['admitted'] = dept.applied * dept.rate
print('within each department, women do better:')
print(dept.pivot(index='dept', columns='sex', values='rate'))
print()
overall = dept.groupby('sex').apply(lambda g: g.admitted.sum()/g.applied.sum(),
                                    include_groups=False)
print('overall, women do worse:')
print(overall.round(3))
print()
print('Both are true. More women applied to the harder department.')


## 9. Mean vs median on skewed data


In [ ]:
revenue = rng.lognormal(4, 1.2, 5000)
print(f'mean   : {revenue.mean():.2f}')
print(f'median : {np.median(revenue):.2f}')
print(f'skew   : {pd.Series(revenue).skew():.2f}')
print()
print('When these differ a lot, report both and say which you used.')


## Try it yourself

1. Bootstrap a confidence interval for the *difference* between two group means.
2. Change the true difference in section 6 to zero and check the p-values stay uniform.
3. Apply a Bonferroni correction (divide alpha by 20) in section 7 and recount.
